In [155]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Cleaning_immo") \
    .master("local[*]") \
    .getOrCreate()

print("Spark pret. Version :", spark.version)

Spark pret. Version : 4.1.1


In [156]:
schema_local = StructType([
    StructField("titre", StringType()),
    StructField("description", StringType()),
    StructField("ville", StringType()),
    StructField("type_bien", StringType()),
    StructField("etage", IntegerType()),
    StructField("prix", DoubleType()),
    StructField("surface", DoubleType()),
    StructField("pieces", IntegerType()),
    StructField("DPE", StringType()),
    StructField("full_description", StringType()),
    StructField("lien", StringType())
])

In [157]:
df_csv = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .schema(schema_local) \
    .csv("annonces_rhone.csv")

print("Nombre de lignes lues :", df_csv.count())
df_csv.show(5, truncate=False)

Nombre de lignes lues : 580
+-----------------------------------------------+----------------------------------------------------------------------------------------------+----------------------+-----------+-----+--------+-------+------+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [158]:
df_csv.printSchema()
print("Types detectes :", df_csv.dtypes)

root
 |-- titre: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- etage: integer (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- DPE: string (nullable = true)
 |-- full_description: string (nullable = true)
 |-- lien: string (nullable = true)

Types detectes : [('titre', 'string'), ('description', 'string'), ('ville', 'string'), ('type_bien', 'string'), ('etage', 'int'), ('prix', 'double'), ('surface', 'double'), ('pieces', 'int'), ('DPE', 'string'), ('full_description', 'string'), ('lien', 'string')]


In [159]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

print("Nombre de lignes avant nettoyage :", df_csv.count())

Valeurs nulles par colonne :
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|    0|          0|    0|        0|  197|   0|      0|     7| 58|               9|   0|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



Nombre de lignes avant nettoyage : 580


In [160]:
df_csv.filter(
    (F.col("etage").isNull()) &
    (F.col("full_description").contains("étage"))
).select("full_description").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [161]:
df_csv.filter(
    (F.col("DPE").isNull()) &
    (F.lower(F.col("full_description").contains("DPE")))
).select("full_description").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [162]:
df_csv = df_csv.withColumn(
    "DPE",
    F.when(
        (F.col("DPE").isNull()) &
        (F.lower(F.col("full_description")).contains("non soumis au dpe")),
        "Non soumis"
    ).when(
        (F.col("DPE").isNull()) &
        (F.lower(F.col("full_description")).contains("dpe a")),
        "A"
    ).when(
        (F.col("DPE").isNull()) &
        (F.lower(F.col("full_description")).contains("dpe : d")),
        "D"
    ).when(
        (F.col("DPE").isNull()) & ( 
            (F.lower(F.col("full_description")).contains("dpe en cours")) | 
            (F.lower(F.col("full_description")).contains("dpe non communiqu")) |
            (F.lower(F.col("full_description")).contains("dpe : non communiqu")) |
            (F.lower(F.col("full_description")).contains("dpe vierge")) |
            (F.lower(F.col("full_description")).contains("dpe est vierge")) |
            (~F.lower(F.col("full_description")).contains("dpe")) |
            (F.col("full_description").isNull())
        ),
        "Inconnu"
    ).otherwise(F.col("DPE"))
)

In [163]:
df_csv.filter(
    (F.col("DPE").isNull()) 
).select("full_description").show(truncate=False)

+----------------+
|full_description|
+----------------+
+----------------+



In [164]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

Valeurs nulles par colonne :
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|    0|          0|    0|        0|  197|   0|      0|     7|  0|               9|   0|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



In [165]:
df_csv.filter(
    (F.col("full_description").isNull()) 
).select("description").show(truncate=False)

+-----------------------------------------------------------------------------------------------------+
|description                                                                                          |
+-----------------------------------------------------------------------------------------------------+
|6732- VILLEFRANCHE S/S Centre ville                                                                  |
|Foch - Bourgeois 152 m2 à rénover                                                                    |
|Appartement 53 m2 entièrement rénové                                                                 |
|Grande piece de vie pour ce t2 au centre du village                                                  |
|Exclusivité T5 de 123M2 EN  REZ  DE JARDIN Charbonnieres  LES BAINS                                  |
|Lyon 1 - place chazette                                                                              |
|Lyon 05, 3 pièces lumineux avec une vue dégagée, dans un parc a

In [166]:
df_csv = df_csv.withColumn(
    "full_description",
    F.when(
        (F.col("full_description").isNull()),
        F.col("description")
    ).otherwise(F.col("full_description"))
)

In [167]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

Valeurs nulles par colonne :
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|    0|          0|    0|        0|  197|   0|      0|     7|  0|               0|   0|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



In [168]:
df_csv.filter(
    (F.col("pieces").isNull()) 
).select("type_bien", "surface").show(truncate=False)

+-----------+-------+
|type_bien  |surface|
+-----------+-------+
|Appartement|461.0  |
|Maison     |104.0  |
|Maison     |135.0  |
|Appartement|375.0  |
|Maison     |336.0  |
|Appartement|400.0  |
|Appartement|251.0  |
+-----------+-------+



In [169]:
df_ratio = df_csv.filter(
    (F.col("surface").isNotNull()) &
    (F.col("pieces").isNotNull()) &
    (F.col("pieces") > 0)
).withColumn(
    "ratio_surface_piece",
    F.col("surface") / F.col("pieces")
)

df_ratio = df_ratio.withColumn(
    "surface_bin",
    F.when(F.col("surface") < 150, "100-150")
     .when(F.col("surface") < 250, "150-250")
     .otherwise("250+")
)


df_ratio.groupBy("type_bien", "surface_bin").agg(
    F.avg("ratio_surface_piece").alias("moyenne"),
    F.expr("percentile_approx(ratio_surface_piece, 0.5)").alias("mediane"),
    F.count("*").alias("nb_obs")
).orderBy("type_bien", "surface_bin").show()

+-----------+-----------+------------------+------------------+------+
|  type_bien|surface_bin|           moyenne|           mediane|nb_obs|
+-----------+-----------+------------------+------------------+------+
|Appartement|    100-150|23.328858439927682|22.666666666666668|   406|
|Appartement|    150-250|30.203246753246756|              30.4|    11|
|Appartement|       250+|54.531746031746025| 47.57142857142857|     4|
|     Maison|    100-150|23.369008799171844|23.333333333333332|    92|
|     Maison|    150-250|28.261702127659575|27.833333333333332|    47|
|     Maison|       250+|  38.8699716949717|              38.8|    13|
+-----------+-----------+------------------+------------------+------+



In [170]:
df_csv = df_csv.withColumn(
    "pieces",
    F.when( 
        (F.col("pieces").isNull()) &
        (F.col("type_bien") == "Appartement") & (F.col("surface") < 150),
        F.round(F.col("surface") / 22.67)
    ).when(
        (F.col("pieces").isNull()) &
        (F.col("type_bien") == "Appartement") & (F.col("surface") < 250),
        F.round(F.col("surface") / 30.4)
    ).when(
        (F.col("pieces").isNull()) &
        (F.col("type_bien") == "Appartement") & (F.col("surface") >=  250),
        F.round(F.col("surface") / 47.57)
    ).when(
        (F.col("pieces").isNull()) &
        (F.col("type_bien") == "Maison") & (F.col("surface") < 150),
        F.round(F.col("surface") / 23.33)
    ).when(
        (F.col("pieces").isNull()) &
        (F.col("type_bien") == "Maison") & (F.col("surface") < 250),
        F.round(F.col("surface") / 27.83)
    ).when(
        (F.col("pieces").isNull()) &
        (F.col("type_bien") == "Maison") & (F.col("surface") >=  250),
        F.round(F.col("surface") / 38.8)
    ).otherwise(F.col("pieces"))
)

In [171]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

Valeurs nulles par colonne :
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|    0|          0|    0|        0|  197|   0|      0|     0|  0|               0|   0|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



In [172]:
df_csv.groupBy("type_bien").count().show()

+-----------+-----+
|  type_bien|count|
+-----------+-----+
|Appartement|  425|
|     Maison|  155|
+-----------+-----+



In [173]:
df_csv = df_csv.withColumn(
    "etage",
    F.when(
        F.lower(F.col("full_description")).contains("dernier étage") | 
        F.lower(F.col("full_description")).contains("dernier etage"), -1
     ).otherwise(F.col("etage"))
)

In [174]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

Valeurs nulles par colonne :
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|    0|          0|    0|        0|  186|   0|      0|     0|  0|               0|   0|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



In [175]:
df_csv = df_csv.withColumn(
    "surface_bin",
        F.when(F.col("surface") < 40, "<40")
         .when((F.col("surface") >= 40) & (F.col("surface") < 70), "40-70")
         .when((F.col("surface") >= 70) & (F.col("surface") < 100), "70-100")
         .when((F.col("surface") >= 100) & (F.col("surface") < 150), "100-150")
         .otherwise("150+")
)

df_csv.groupBy("type_bien", "surface_bin", "etage") \
  .count() \
  .orderBy("type_bien", "surface_bin", F.desc("count")) \
  .show()

+-----------+-----------+-----+-----+
|  type_bien|surface_bin|etage|count|
+-----------+-----------+-----+-----+
|Appartement|    100-150| NULL|   21|
|Appartement|    100-150|   -1|   11|
|Appartement|    100-150|    1|   10|
|Appartement|    100-150|    3|    6|
|Appartement|    100-150|    2|    5|
|Appartement|    100-150|    6|    3|
|Appartement|    100-150|    0|    2|
|Appartement|    100-150|    8|    2|
|Appartement|    100-150|    9|    1|
|Appartement|       150+| NULL|    9|
|Appartement|       150+|    1|    5|
|Appartement|       150+|    0|    2|
|Appartement|       150+|   -1|    1|
|Appartement|       150+|    2|    1|
|Appartement|       150+|    4|    1|
|Appartement|      40-70| NULL|   43|
|Appartement|      40-70|    1|   37|
|Appartement|      40-70|   -1|   21|
|Appartement|      40-70|    2|   15|
|Appartement|      40-70|    4|   12|
+-----------+-----------+-----+-----+
only showing top 20 rows


In [176]:
df_filtered = df_csv.filter(F.col("etage").isNotNull())

w = Window.partitionBy("type_bien", "surface_bin") \
          .orderBy(F.desc("count"))

mode_df = df_filtered.groupBy("type_bien", "surface_bin", "etage") \
            .count() \
            .withColumn("rn", F.row_number().over(w)) \
            .filter(F.col("rn") == 1)
mode_df.show()

+-----------+-----------+-----+-----+---+
|  type_bien|surface_bin|etage|count| rn|
+-----------+-----------+-----+-----+---+
|Appartement|    100-150|   -1|   11|  1|
|Appartement|       150+|    1|    5|  1|
|Appartement|      40-70|    1|   37|  1|
|Appartement|     70-100|    1|   21|  1|
|Appartement|        <40|    1|   16|  1|
|     Maison|    100-150|    0|   24|  1|
|     Maison|       150+|    0|   15|  1|
|     Maison|      40-70|    1|    2|  1|
|     Maison|     70-100|    0|    9|  1|
+-----------+-----------+-----+-----+---+



In [177]:
df_csv = df_csv.withColumn(
    "etage",
        F.when((F.col("etage").isNull()) &(F.col("type_bien") =="Appartement") &
               ((F.col("surface") <100) | (F.col("surface") >= 100)), 1
        ).when((F.col("etage").isNull()) &(F.col("type_bien") =="Appartement") &
               (F.col("surface_bin") =="100-150"), -1
        ).when((F.col("etage").isNull()) &(F.col("type_bien") =="Maison"),0
        ).otherwise(F.col("etage"))
)

In [178]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

Valeurs nulles par colonne :
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+-----------+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|surface_bin|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+-----------+
|    0|          0|    0|        0|    0|   0|      0|     0|  0|               0|   0|          0|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+-----------+



In [179]:
numeric_cols = [
    f.name for f in df_csv.schema.fields
    if isinstance(f.dataType, (IntegerType, DoubleType))
]

for c in numeric_cols:
    if c not in ["prix"]:
        p99 = df_csv.approxQuantile(c, [0.99], 0.01)[0]

        print(f"\n=== {c} > 99e percentile ({p99}) ===")

        df_csv.filter(F.col(c) >= p99).show(truncate=False)


=== etage > 99e percentile (13.0) ===
+------------------------------------+--------------------------------------------------------------------+------------+-----------+-----+--------+-------+------+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [180]:
df_csv = df_csv.drop("surface_bin")

In [181]:
df_csv = df_csv.withColumn("prix_m2", F.col("prix") / F.col("surface"))
df_csv = df_csv.filter((F.col("prix_m2") >= 500) & (F.col("prix_m2") <= 25000))
df_csv = df_csv.filter((F.col("prix_m2") >= 500) & (F.col("prix_m2") <= 25000))
df_csv = df_csv.filter((F.col("surface") <925))
df_csv = df_csv.drop("prix_m2")

In [182]:
df_csv.filter((F.col("surface") >=925)).show()


+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



In [183]:
df_csv.filter((F.col("prix_m2") < 500) & (F.col("prix_m2") > 25000)).show()


+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
|titre|description|ville|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+
+-----+-----------+-----+---------+-----+----+-------+------+---+----------------+----+



In [184]:
df_csv.toPandas().to_csv("etreproprio_rhone_propre.csv", index=False, encoding="utf-8-sig")